# Running and comparing neural audio codecs

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/dorienh/music-ai-explained/blob/main/03-comparing-codecs/notebook.ipynb)

Companion to the article *I Ran Five Neural Audio Codecs on the Same Music*. We run EnCodec, DAC, SNAC and Mimi, look at the integers they produce, compute the standard metrics, and listen.

Runs on a CPU. The first run downloads the model weights (about 1 GB in total). The full 13-system benchmark from the article is in `codec_benchmark.py` in the same folder.

In [1]:
import sys
if "google.colab" in sys.modules:
    !pip install -q librosa soundfile encodec descript-audio-codec snac "transformers>=4.45" pesq pystoi

In [2]:
import warnings; warnings.filterwarnings("ignore")
import numpy as np, librosa, matplotlib.pyplot as plt
from IPython.display import Audio, display

BLUE, ORANGE, AQUA, INK2, GRID = "#2a78d6", "#eb6834", "#1baf7a", "#52514e", "#e4e3df"
plt.rcParams.update({"axes.spines.top": False, "axes.spines.right": False, "axes.grid": True,
                     "grid.color": GRID, "lines.linewidth": 2, "figure.dpi": 110})

def clip(name, sr=None, offset=0.0, duration=None):
    """Load one of librosa's freely licensed example recordings (downloaded on first use)."""
    y, sr = librosa.load(librosa.example(name, hq=True), sr=sr, mono=True, offset=offset, duration=duration)
    return y, sr
import torch, time
torch.set_grad_enabled(False)

def resample(y, a, b):
    return y if a == b else librosa.resample(y, orig_sr=a, target_sr=b)

def to_tensor(y):                                  # codecs expect [batch, channels, samples]
    return torch.from_numpy(y)[None, None]

music, MSR = clip("fishin", offset=30, duration=10)        # a singer with a band, 44.1 kHz
speech, SSR = clip("libri1", duration=10)                  # read speech, 16 kHz
print("music:", MSR, "Hz   speech:", SSR, "Hz")

music: 44100 Hz   speech: 16000 Hz


## 1. The codecs

Each function takes audio and its sample rate and returns the decoded audio, its sample rate, and the codes.

### EnCodec (24 kHz, 75 frames per second)

In [3]:
from encodec import EncodecModel
encodec_model = EncodecModel.encodec_model_24khz().eval()

def run_encodec(y, sr, kbps=6.0):
    encodec_model.set_target_bandwidth(kbps)               # 1.5, 3, 6, 12 or 24
    frames = encodec_model.encode(to_tensor(resample(y, sr, 24000)))
    codes = torch.cat([c for c, _ in frames], dim=-1)      # [1, codebooks, frames]
    return encodec_model.decode(frames)[0, 0].numpy(), 24000, codes

out, sr_out, codes = run_encodec(music, MSR)
print("codes:", tuple(codes.shape), " values from", codes.min().item(), "to", codes.max().item())
print("tokens back to audio, first 2 codebooks only:", tuple(encodec_model.decode([(codes[:, :2], None)]).shape))

codes: (1, 8, 751)  values from 0 to 1023
tokens back to audio, first 2 codebooks only: (1, 1, 240320)


### DAC (44.1 kHz, 86 frames per second, 9 codebooks)

In [4]:
import dac
dac_model = dac.DAC.load(dac.utils.download(model_type="44khz")).eval()

def run_dac(y, sr, n_codebooks=None):
    x = dac_model.preprocess(to_tensor(resample(y, sr, 44100)), 44100)
    z, codes, latents, _, _ = dac_model.encode(x, n_quantizers=n_codebooks)
    if n_codebooks: codes = codes[:, :n_codebooks]
    return dac_model.decode(z)[0, 0].numpy(), 44100, codes

x = dac_model.preprocess(to_tensor(music), 44100)
z, codes, latents, _, _ = dac_model.encode(x)
print("codes:  ", tuple(codes.shape), "  the integers")
print("z:      ", tuple(z.shape), "  sum of the looked-up entries, what the decoder consumes")
print("latents:", tuple(latents.shape), "  9 codebooks x 8 numbers: the small space where the lookup happens")
z3 = dac_model.quantizer.from_codes(codes[:, :3])[0]      # tokens back to audio with 3 codebooks
print("decoded from 3 codebooks:", tuple(dac_model.decode(z3).shape))

codes:   (1, 9, 862)   the integers
z:       (1, 1024, 862)   sum of the looked-up entries, what the decoder consumes
latents: (1, 72, 862)   9 codebooks x 8 numbers: the small space where the lookup happens


decoded from 3 codebooks: (1, 1, 441344)


### SNAC (multi-scale: each codebook has its own frame rate)

In [5]:
from snac import SNAC
snac_model = SNAC.from_pretrained("hubertsiuzdak/snac_44khz").eval()

def run_snac(y, sr):
    codes = snac_model.encode(to_tensor(resample(y, sr, 44100)))   # a list, one tensor per codebook
    return snac_model.decode(codes)[0, 0].numpy(), 44100, codes

_, _, codes = run_snac(music, MSR)
print("tokens per codebook:", [c.shape[-1] for c in codes], " per second:", [round(c.shape[-1] / 10, 1) for c in codes])

tokens per codebook: [144, 288, 576, 1152]  per second: [14.4, 28.8, 57.6, 115.2]


### Mimi (speech, 12.5 frames per second)

The released model has 32 codebooks. The Moshi speech model uses 8, so pass `num_quantizers=8`.

In [6]:
from transformers import MimiModel
mimi_model = MimiModel.from_pretrained("kyutai/mimi").eval()

def run_mimi(y, sr, n_codebooks=8):
    enc = mimi_model.encode(to_tensor(resample(y, sr, 24000)), num_quantizers=n_codebooks)
    return mimi_model.decode(enc.audio_codes)[0][0, 0].numpy(), 24000, enc.audio_codes

_, _, codes = run_mimi(speech, SSR)
print("codes:", tuple(codes.shape), " default without num_quantizers:", tuple(mimi_model.encode(to_tensor(resample(speech, SSR, 24000))).audio_codes.shape))

codes: (1, 8, 125)  default without num_quantizers: (1, 32, 125)


### Baseline: Opus and MP3 through ffmpeg

In [7]:
import shutil, subprocess, os, soundfile as sf
HAVE_FFMPEG = shutil.which("ffmpeg") is not None

def run_ffmpeg(y, sr, codec, kbps, ext):
    in_sr = 48000 if codec == "libopus" else 44100
    sf.write("_in.wav", resample(y, sr, in_sr), in_sr)
    subprocess.run(["ffmpeg", "-y", "-loglevel", "error", "-i", "_in.wav", "-c:a", codec, "-b:a", f"{kbps}k", f"_t.{ext}"], check=True)
    subprocess.run(["ffmpeg", "-y", "-loglevel", "error", "-i", f"_t.{ext}", "_out.wav"], check=True)
    out, out_sr = sf.read("_out.wav", dtype="float32")
    return out, out_sr, os.path.getsize(f"_t.{ext}") * 8 / (len(y) / sr) / 1000     # measured kbit/s

## 2. The metrics

**SI-SDR** compares waveforms: how large is the error relative to the signal, after removing any volume difference? By hand, with three samples:

In [8]:
def si_sdr(ref, est):
    ref, est = ref - ref.mean(), est - est.mean()
    target = (est @ ref) / (ref @ ref) * ref          # the scaled copy of ref closest to est
    noise = est - target                              # everything else
    return 10 * np.log10((target @ target) / (noise @ noise))

ref, est = np.array([1., 2., 2.]), np.array([1., 2., 1.])
target = (est @ ref) / (ref @ ref) * ref; noise = est - target
print("target:", target.round(3), " noise:", noise.round(3))
print(f"energy of target {target @ target:.2f}, of noise {noise @ noise:.2f}  ->  {10 * np.log10((target @ target) / (noise @ noise)):.1f} dB (without removing the mean)")

target: [0.778 1.556 1.556]  noise: [ 0.222  0.444 -0.556]
energy of target 5.44, of noise 0.56  ->  9.9 dB (without removing the mean)


**Mel distance** compares log-mel spectrograms and ignores the fine timing of the wave. **PESQ** and **STOI** are models of a listener, built for speech.

In [9]:
from pesq import pesq
from pystoi import stoi

def mel_distance(ref, est, sr):
    def logmel(y):
        m = librosa.feature.melspectrogram(y=y, sr=sr, n_fft=2048, hop_length=512, n_mels=80, power=1.0)
        return np.log10(np.maximum(m, 1e-5))
    return float(np.abs(logmel(ref) - logmel(est)).mean())

def align(ref, est, max_lag=1600):                    # codecs can shift the signal by a few samples
    n = min(len(ref), len(est), 16000 * 5)
    c = np.correlate(est[:n], ref[:n], "full"); lags = np.arange(-n + 1, n); keep = np.abs(lags) <= max_lag
    lag = lags[keep][np.abs(c[keep]).argmax()]
    est = est[lag:] if lag >= 0 else np.concatenate([np.zeros(-lag, dtype=est.dtype), est])
    n = min(len(ref), len(est)); return ref[:n], est[:n]

def score(ref, sr, est, est_sr, is_speech):
    r16, e16 = align(resample(ref, sr, 16000), resample(est, est_sr, 16000))
    row = {"SI-SDR (dB)": round(float(si_sdr(r16, e16)), 1)}
    if is_speech:
        row["PESQ"] = round(pesq(16000, r16, e16, "wb"), 2); row["STOI"] = round(float(stoi(r16, e16, 16000)), 3)
    else:
        full = resample(est, est_sr, sr); n = min(len(ref), len(full))
        row["mel distance, full range"] = round(mel_distance(ref[:n], full[:n], sr), 3)
    return row

## 3. Same audio, every codec

One 10-second clip of music and one of speech. This is a single clip each, so read it as a demonstration of how the metrics behave. The article's table averages three music clips.

In [10]:
import pandas as pd

def kbps_of(codes, seconds, bits):
    n = sum(c.shape[-1] for c in codes) if isinstance(codes, list) else codes.shape[-2] * codes.shape[-1]
    return n * bits / seconds / 1000

systems = [
    ("EnCodec 24k, 2 codebooks",  lambda y, sr: run_encodec(y, sr, 1.5), 10),
    ("EnCodec 24k, 8 codebooks",  lambda y, sr: run_encodec(y, sr, 6.0), 10),
    ("EnCodec 24k, 32 codebooks", lambda y, sr: run_encodec(y, sr, 24.0), 10),
    ("DAC 44k, 3 codebooks",      lambda y, sr: run_dac(y, sr, 3), 10),
    ("DAC 44k, 9 codebooks",      lambda y, sr: run_dac(y, sr), 10),
    ("SNAC 44k",                  run_snac, 12),
    ("Mimi, 8 codebooks",         run_mimi, 11),
]
if HAVE_FFMPEG:
    systems += [("Opus 12k", lambda y, sr: run_ffmpeg(y, sr, "libopus", 12, "opus"), None),
                ("MP3 32k",  lambda y, sr: run_ffmpeg(y, sr, "libmp3lame", 32, "mp3"), None)]

decoded, tables = {}, {}
for label, (ref, sr, is_speech) in {"music": (music, MSR, False), "speech": (speech, SSR, True)}.items():
    rows = []
    for name, fn, bits in systems:
        t = time.time(); est, est_sr, info = fn(ref, sr); seconds = time.time() - t
        kbps = info if bits is None else kbps_of(info, len(ref) / sr, bits)
        rows.append({"system": name, "kbit/s": round(kbps, 2), **score(ref, sr, est, est_sr, is_speech), "real-time factor": round(seconds / (len(ref) / sr), 3)})
        decoded[(label, name)] = (est, est_sr)
    tables[label] = pd.DataFrame(rows).set_index("system")
tables["music"]

,kbit/s,SI-SDR (dB),"mel distance, full range",real-time factor
system,,,,
"EnCodec 24k, 2 codebooks",1.50,2.0,0.422,0.029
"EnCodec 24k, 8 codebooks",6.01,7.5,0.374,0.029
"EnCodec 24k, 32 codebooks",24.03,10.7,0.345,0.030
"DAC 44k, 3 codebooks",2.59,6.2,0.178,0.236
"DAC 44k, 9 codebooks",7.76,12.5,0.121,0.237
SNAC 44k,2.59,6.7,0.154,0.269
"Mimi, 8 codebooks",1.11,1.3,0.462,0.023
Opus 12k,12.51,11.0,0.527,0.036
MP3 32k,32.28,18.0,0.624,0.022


In [11]:
tables["speech"]

,kbit/s,SI-SDR (dB),PESQ,STOI,real-time factor
system,,,,,
"EnCodec 24k, 2 codebooks",1.50,1.9,1.07,0.816,0.031
"EnCodec 24k, 8 codebooks",6.00,8.8,2.84,0.929,0.028
"EnCodec 24k, 32 codebooks",24.00,14.1,3.94,0.977,0.029
"DAC 44k, 3 codebooks",2.59,6.7,2.10,0.885,0.241
"DAC 44k, 9 codebooks",7.76,14.4,3.85,0.967,0.242
SNAC 44k,2.59,6.0,1.11,0.884,0.270
"Mimi, 8 codebooks",1.10,4.1,1.89,0.881,0.022
Opus 12k,12.65,15.2,3.85,0.967,0.029
MP3 32k,32.28,21.8,4.13,0.992,0.021


Two things to look for in the music table. MP3 at 32 kbit/s has a far higher SI-SDR than DAC and a far worse mel distance: the two metrics disagree. And EnCodec at 24 kHz barely improves its full-range mel distance when you give it more codebooks, because it cannot output anything above 12 kHz.

## 4. Listen

No metric replaces this.

In [12]:
print("original"); display(Audio(music, rate=MSR))
for name in ("EnCodec 24k, 2 codebooks", "EnCodec 24k, 8 codebooks", "DAC 44k, 3 codebooks", "DAC 44k, 9 codebooks", "SNAC 44k", "Mimi, 8 codebooks", "MP3 32k"):
    if ("music", name) in decoded:
        est, est_sr = decoded[("music", name)]
        print(name, "|", tables["music"].loc[name, "kbit/s"], "kbit/s"); display(Audio(est, rate=est_sr))

original


[audio player: run the cell to listen]


EnCodec 24k, 2 codebooks | 1.5 kbit/s


[audio player: run the cell to listen]


EnCodec 24k, 8 codebooks | 6.01 kbit/s


[audio player: run the cell to listen]


... (6 outputs left out)

Mimi, 8 codebooks | 1.11 kbit/s


[audio player: run the cell to listen]


MP3 32k | 32.28 kbit/s


[audio player: run the cell to listen]


## Where next

- The articles: [Music & AI on Medium](https://medium.com/@dorien-herremans/list/music-ai-3fb88d4826dd)
- The next notebook: [autoregressive, diffusion and flow matching](../04-autoregressive-diffusion-flow-matching/notebook.ipynb)
- For the full picture: [A Tutorial on Generative Models for Music Audio](https://arxiv.org/abs/2610.38252)